## Data Extraction, Chunk Cleaning and Preparation

In [ ]:
from bs4 import BeautifulSoup
import requests

response = requests.get(r"https://www.consumer.vic.gov.au/housing/renting/rent-bond-bills-and-condition-reports/paying-for-utilities-phone-and-internet")


In [ ]:
html = response.content.decode("utf-8")

html

In [ ]:
with open(r"..\\raw_data\\Paying_for_utilities_and_services.txt", "w", encoding="utf-8") as f:
    f.write(html)

In [ ]:
with open(r"..\\raw_data\\Paying_for_utilities_and_services.txt", "r", encoding="utf-8") as f:
    html = f.read()

In [ ]:
headers_to_split_on = [("h1", "h1"), ("h2", "h2"), ("h3", "h3"), ("h4", "h4"), ("h5", "h5"), ("h6", "h6")]

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_header_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_element=False,
)

header_chunks = html_header_splitter.split_text(html)

In [ ]:
header_chunks

In [ ]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter

def table_to_text(table):
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    lines = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True).replace("\xa0", " ")
                 for td in tr.find_all("td")]
        if cells:
            lines.append("; ".join(f"{h}: {c}" for h, c in zip(headers, cells)))
    return "\n".join(lines)

html_semantic_splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    max_chunk_size=1500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", " "],
    elements_to_preserve=["table", "ul", "ol"],
    custom_handlers={"table": table_to_text},
    external_metadata= {"url": "the_url"}, # We add the url as we will implement RAG with Sources
    preserve_links=False,
)

semantic_chunks = html_semantic_splitter.split_text(html)

In [ ]:
semantic_chunks

The structure of the web pages is such that we can advantage of it.

From the structure of the web pages on the website, we can also glean that all contents before the first `<h1>` header is mostly unrelated content. We can clean the chunks even more by eliminating the 'Sharethis page' chunk, which is a common noisy chunk in all the web pages.

All elements in the footer (starting with the chunk 'About us') is mostly noise.

Hence we can do some post-processing to eliminate such chunks.

In [ ]:
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

def clean_chunks(semantic_chunks):
    new_chunks = []
    header_1_met = False
    share_chunk_met = False
    footer_reached = False

    for i in range(0, len(semantic_chunks)):
        chunk = semantic_chunks[i]
        share_this_page_chunk = False
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                    if not header_1_met and HEADER_KEY.match(key):
                            if key == "h1":
                                header_1_met = True
                                break

                    if not share_chunk_met and value == "Sharethis page":
                            share_chunk_met = True
                            new_chunks[-1].page_content = chunk.page_content[20:].strip()
                            share_this_page_chunk = True
                            break

                    if key == "h3" and value == "About us":
                          footer_reached = True
                          break

        # If we reached the footer, there is no need to continue
        if footer_reached:
              break

        # Skip the chunk if we have not encountered our first header yet
        if not header_1_met or share_this_page_chunk:
             continue

        # Remove meaningless encodings from the chunk page_content
        chunk.page_content.replace("\xa0", " ")

        # Add the chunk
        new_chunks.append(chunk)

    return new_chunks

cleaned_semantic_chunks = clean_chunks(semantic_chunks)

In [ ]:
cleaned_semantic_chunks

We can also see that it is better to use the HTMLSemanticPreservingSplitter because of its 'custom_handlers' attribute that allows us to reformat table data and possibly other types of data in a more semantically meaningful way.

I also believe that HTMLSemanticPreservingSplitter will perform better than true Semantic Chunking because the structure of the web pages are very well-defined.

One big issue with HTMLSemanticPreservingSplitter is that each chunk has only its parent heading in its metadata, not all its ancestor heading. This can make our chunks lose meaningful context. In such case, the type of header metadata for headers provided by HTMLHeaderTextSplitter is highly preferred.

Direct metadata filtering on the headers is not reasonable as the header text is not exactly unique (meaning they could be written in multiple ways by the user, unlike product codes).

The best way to handle this is to preprocess the chunks so that the header tags are present as in HTMLHeaderTextSplitter and append all ancestor headers to the content as a prefix.

In [ ]:
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

def post_process_header_tags(semantic_chunks):
    # List to contain header texts of h1-h6.
    headers = []
    # Variable to hold current smallest header traversed
    curr_smallest_header_traversed = 0
    for chunk in semantic_chunks:
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                if HEADER_KEY.match(key):
                    header_index = int(key[-1])

                    # Adjust the header hierarchy for the chunk
                    if header_index == curr_smallest_header_traversed:
                        headers.pop()
                        headers.append((header_index,value))
                    elif header_index > curr_smallest_header_traversed:
                        curr_smallest_header_traversed = header_index
                        headers.append((header_index, value))
                    else:
                        curr_smallest_header_traversed = header_index
                        for i in range(1, len(headers) + 1):
                            if headers[-1][0] >= header_index:
                                headers.pop()
                            else:
                                break

                        headers.append((header_index, value))

                    chunk_header = ""

                    # Create the chunk header string
                    for header in headers:
                        chunk_header = chunk_header + header[1] + " - "

                    # Append all ancestor headers to the chunk content
                    chunk.page_content = chunk_header + chunk.page_content

In [ ]:
post_process_header_tags(cleaned_semantic_chunks)


In [ ]:
cleaned_semantic_chunks

## Assesssing quality of retrieval with current chunking logic

Now, let us check how good vector similarity search works on our extracted chunks

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from load_dotenv import load_dotenv
import os

# Load our environment files into the environment
load_dotenv()

# We will use 'text-embedding-3-small' OpenAI model as both our prompt encoder and context encoder
encoder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

if not os.path.exists("./chroma_db"):
    vector_store = Chroma.from_documents(
        documents=cleaned_semantic_chunks,
        embedding=encoder,
        persist_directory="./chroma_db"
    )
else:
    vector_store = Chroma(
        persist_directory="./chroma_db",
        embedding_function=encoder
    )

In [ ]:
query = "What should a renter do when he moves out?"

In [ ]:
results = vector_store.similarity_search(
    query=query,
    k=4
)

In [ ]:
for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

We can see that the retriever returned the most relevant content from the vector store as well as some noisy chunks. However, the most relevant content was indeed ranked as first.

Let us try a more complex query.

In [ ]:
query = "I want to install a phone connection in my apartment. What steps should I take?"

In [ ]:
results = vector_store.similarity_search(
    query=query,
    k=4
)

for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

Again, the results are highly satisfactory. The retriever returned the most relevant chunks with the first one being exactly what we need.

However, we have been testing with data from only 1 web page. Let us try fetching multiple pages and querying the relevant sections.

## Testing with Multiple Web Pages' Content

In [2]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

headers_to_split_on = [("h1", "h1"), ("h2", "h2"), ("h3", "h3"), ("h4", "h4"), ("h5", "h5"), ("h6", "h6")]

def table_to_text(table):
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    lines = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True).replace("\xa0", " ")
                 for td in tr.find_all("td")]
        if cells:
            lines.append("; ".join(f"{h}: {c}" for h, c in zip(headers, cells)))
    return "\n".join(lines)

html_semantic_splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    max_chunk_size=1500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", " "],
    elements_to_preserve=["table", "ul", "ol"],
    custom_handlers={"table": table_to_text},
    external_metadata= {"url": "the_url"}, # We add the url as we will implement RAG with Sources
    preserve_links=False,
)

def clean_chunks(semantic_chunks):
    new_chunks = []
    header_1_met = False
    share_chunk_met = False
    footer_reached = False

    for i in range(0, len(semantic_chunks)):
        chunk = semantic_chunks[i]
        share_this_page_chunk = False
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                    if not header_1_met and HEADER_KEY.match(key):
                            if key == "h1":
                                header_1_met = True
                                break

                    if not share_chunk_met and value == "Sharethis page":
                            share_chunk_met = True
                            new_chunks[-1].page_content = chunk.page_content[20:].strip()
                            share_this_page_chunk = True
                            break

                    if key == "h3" and value == "About us":
                          footer_reached = True
                          break

        # If we reached the footer, there is no need to continue
        if footer_reached:
              break

        # Skip the chunk if we have not encountered our first header yet
        if not header_1_met or share_this_page_chunk:
             continue

        # Remove meaningless encodings from the chunk page_content
        chunk.page_content.replace("\xa0", " ")

        # Add the chunk
        new_chunks.append(chunk)

    return new_chunks

def post_process_header_tags(semantic_chunks):
    # List to contain header texts of h1-h6.
    headers = []
    # Variable to hold current smallest header traversed
    curr_smallest_header_traversed = 0
    for chunk in semantic_chunks:
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                if HEADER_KEY.match(key):
                    header_index = int(key[-1])

                    # Adjust the header hierarchy for the chunk
                    if header_index == curr_smallest_header_traversed:
                        headers.pop()
                        headers.append((header_index,value))
                    elif header_index > curr_smallest_header_traversed:
                        curr_smallest_header_traversed = header_index
                        headers.append((header_index, value))
                    else:
                        curr_smallest_header_traversed = header_index
                        for i in range(1, len(headers) + 1):
                            if headers[-1][0] >= header_index:
                                headers.pop()
                            else:
                                break

                        headers.append((header_index, value))

                    chunk_header = ""

                    # Create the chunk header string
                    for header in headers:
                        chunk_header = chunk_header + header[1] + " - "

                    # Append all ancestor headers to the chunk content
                    chunk.page_content = chunk_header + chunk.page_content

In [3]:
from glob import glob

web_files = glob("..\\raw_data\\*.txt")

web_files

['..\\raw_data\\Different_rental_agreements.txt',
 '..\\raw_data\\Finding_and_applying_for_a_rental_property.txt',
 '..\\raw_data\\Introduction_and_helpful_contacts.txt',
 '..\\raw_data\\Paying_for_utilities_and_services.txt',
 '..\\raw_data\\Residential_rental_agreements.txt']

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Create the encoder
encoder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Create the new empty Chroma vector store
vector_store = Chroma(
    embedding_function=encoder,
)

for file_path in web_files:

    with open(file_path, "r") as f:
        html = f.read()

    semantic_chunks = html_semantic_splitter.split_text(html)
    cleaned_semantic_chunks = clean_chunks(semantic_chunks)
    post_process_header_tags(cleaned_semantic_chunks)

    vector_store.add_documents(cleaned_semantic_chunks)

In [16]:
cleaned_semantic_chunks

[Document(metadata={'h1': 'Residential rental agreements', 'url': 'the_url'}, page_content='Residential rental agreements - On this page: Rental agreements What should be in a rental agreement Different types of residential rental agreements Before moving in Changing agreements Conditions that cannot be in an agreement Forms you might need renewing agreements notices to vacate renters giving notice Other pages have information about , , and .'),
 Document(metadata={'h2': 'Rental agreements', 'url': 'the_url'}, page_content='Residential rental agreements - Rental agreements - 3 types of rental agreements A rental agreement used to be called a lease or a residential tenancy agreement. It is a contract between the renter (tenant) and rental provider (landlord). There are for residential properties such as houses, units and apartments. rooming houses caravan parks site tenants in residential parks and villages There are different agreements for , , and .'),
 Document(metadata={'h2': 'What 

In [5]:
def get_results(query, k):
    results = vector_store.similarity_search(
        query=query,
        k=4
    )

    for i, doc in enumerate(results):
        print(f"=== Doc {i+1} ===")
        print(doc.page_content)
        print(doc.metadata)
        print("\n")

In [6]:
query = "What documents must I provide to my agent/renter when applying for a rental property?"

get_results(query, 4)

=== Doc 1 ===
Finding and applying for a rental property - Documents and information you may need to provide: - Formal identification (such as a driver’s licence) Employment and financial details,  such as a payslip Rental or residential history Personal references, if requested. Rental providers must ask for this information through the prescribed rental application form. They can’t ask you for any information that is not set out in the form. applying for a property Read more about . The rental provider or agent cannot ask you for the following information in your application: Whether you’ve taken legal action or had a dispute with a previous rental provider. Your bond history, including any claims you’ve made on your bond. Detailed bank statements with daily transactions (if you need to provide a statement, you can delete transactions you think should be kept private).
{'h3': 'Documents and information you may need to provide:', 'url': 'the_url'}


=== Doc 2 ===
Residential rental ag

In [7]:
query = "I am new to Victoria. Who should I be calling to know more about the renting laws here?"

get_results(query, 4)

=== Doc 1 ===
Introduction and helpful contacts - Introduction - Victoria has some of the strongest rental protections in Australia, but renting can still be complicated. With 1 in 3 Victorians now renting, understanding your rental rights has never been more important. Renters Guide The explains the rules and your rights at every stage of renting in Victoria, from applying for a property to moving out. Whether you’re a first-time renter or have rented before, this guide provides you with the information you need to rent confidently.
{'url': 'the_url', 'h3': 'Introduction'}


=== Doc 2 ===
Finding and applying for a rental property -  - Learn about finding and applying for a rental property in Victoria:
{'url': 'the_url', 'h3': ''}


=== Doc 3 ===
Introduction and helpful contacts - Stay updated - Renting For the most current information, go to . Facebook Instagram Follow Consumer Affairs Victoria on and for regular renting information and updates.
{'url': 'the_url', 'h3': 'Stay update

This one's result is not perfect. We might consider RAG optimisation techniques depending on our later recall rate. Let us try another one.

In [9]:
query = "My renter increased my rent without telling me. I believe this is not allowed. Who should I contact to clarify this?"

get_results(query, 4)

=== Doc 1 ===
Residential rental agreements - Before moving in - Report an issue with an advertised rental property - Report an issue with an advertised rental property Prospective renters with concerns about an advertised rental property can report an issue to us. Find out more, go to .
{'h3': 'Report an issue with an advertised rental property', 'url': 'the_url'}


=== Doc 2 ===
Paying for utilities and services - Very high usage charges - If a renter gets a very high bill for using water, gas or electricity because there is a hidden fault at the property, the rental provider might have to pay the excess amount. These very high bills are also called excessive usage charges. The rental provider only has to pay excessive usage charges if: the renter notified the rental provider, as soon as possible, of the excessive charges and the fault that caused them the renter didn’t cause the fault, and the service provider isn’t responsible for the fault. The rental provider must also reimburse 

We can see that as the complexity of the query increases, the recall rate decreases.

In [8]:
query = "I have questions about renting agreements and bonds. Who should I contact?"

get_results(query, 4)

=== Doc 1 ===
Introduction and helpful contacts - Key terms - Rental provider: landlord. Renter: tenant. Agent: real estate agent. The Residential Tenancies Act 1997: This sets out the laws, rights and responsibilities of renters and rental providers in Victoria. RTBA: the Residential Tenancies Bond Authority. It holds residential rental bonds for Victorians as a neutral party, for the renter and the rental provider. PRBS: Portable Rental Bond Scheme. Eligible renters can transfer the value of their existing bond or ‘bond money’ to their next rental property, saving them being out of pocket. RDRV: Rental Dispute Resolution Victoria. A free dispute resolution service to help renters and rental providers resolve disputes. VCAT: Victorian Civil and Administrative Tribunal. This tribunal resolves legal disputes and cases in Victoria, including renting issues.
{'h3': 'Key terms', 'url': 'the_url'}


=== Doc 2 ===
Finding and applying for a rental property - Documents and information you may

Putting all the table rows as a single Lanchain Document object seems to dilute the context. It would be better to create a separate Document object for each row of the table.

## Changing Chunking Logic to represent each row of a table as a separate Document Object

I think it is better to change the logic to represent each row of a table as a separate langchain_core.documents.Document object. Moreover, logic must be introduced to handle links properly which the previous logic totally misses.

In [45]:
from bs4 import BeautifulSoup

with open(r"..\raw_data\Residential_rental_agreements.txt", "r") as f:
    html = f.read()

html

'\n\n<!DOCTYPE HTML>\n\n\n\n<html class="no-js" lang="en">\n\n\n\n  <head><meta charset="utf-8" /><meta http-equiv="X-UA-Compatible" content="IE=edge" />\n\n\n\n    <!-- Google Tag Manager Data Layer -->\n\n    <script>\n\n        window.dataLayer = [\n\n        {\n\n//          \'countryCode\' : \'{{Country Code}}\', // AU\n\n            \'language\' : \'\', // English\n\n            \'environment\' : \'Production\',\n\n            \'currentVersion\' : \'1.0.0.0\', // 1.0\n\n            \'pageType\' : \'Content Page\', // Home / Product / Category etc\n\n//          \'uiExperience\' : \'{{User Interface}}\', // Desktop / Mobile\n\n            \'user\' : {\n\n                \'id\' : \'\', // 0\n\n                \'type\' : \'Guest\' // Guest\n\n            },\n\n//          \'word-count-range\': {{Word Range}}, \n\n//          \'word-count\': {{Words}},\n\n            \'session-start\': \'1795520861\',\n\n            \'emergency\': document.querySelector(\'aside.notification-bar\') !=

In [46]:
from bs4 import BeautifulSoup
from langchain_core.documents import Document

HEADINGS = ["h1", "h2", "h3", "h4", "h5", "h6"]
INLINE_TAGS = ["a", "span", "strong", "b", "em", "i", "u", "abbr"]
EXCLUDED_HEADINGS = {"share this page"}


def clean(text):
    return " ".join(text.replace("\xa0", " ").split())


def cell_text(cell):
    for tag in cell.find_all(INLINE_TAGS):
        tag.unwrap()
    cell.smooth()
    return clean(cell.get_text(" ", strip=True))


def heading_hierarchy(table):
    hierarchy = {}
    current_level = 7
    for h in table.find_all_previous(HEADINGS):
        level = int(h.name[1])
        if level < current_level:
            hierarchy[f"h{level}"] = clean(h.get_text(" ", strip=True))
            current_level = level
            if level == 1:
                break
    return dict(sorted(hierarchy.items()))


def table_to_docs(table, url):
    hierarchy = heading_hierarchy(table)

    hierarchy = {
        level: text
        for level, text in hierarchy.items()
        if text.lower() not in EXCLUDED_HEADINGS
    }

    prefix = " - ".join(hierarchy.values())

    metadata_base = {"url": url}
    if hierarchy:
        deepest = list(hierarchy)[-1]
        metadata_base = {deepest: hierarchy[deepest], "url": url}

    rows = table.find_all("tr")
    if not rows:
        return []
    headers = [cell_text(c) for c in rows[0].find_all(["th", "td"])]

    docs = []
    for tr in rows[1:]:
        cells = [cell_text(c) for c in tr.find_all(["td", "th"])]
        if len(cells) != len(headers):
            print(f"Skipping irregular row (colspan/rowspan?): {cells}")
            continue
        body = "; ".join(f"{h}: {v}" for h, v in zip(headers, cells))
        page_content = f"{prefix} - {body}" if prefix else body
        docs.append(Document(page_content=page_content, metadata=dict(metadata_base)))
    return docs


def extract_table_docs(html, url):
    soup = BeautifulSoup(html, "html.parser")
    body = soup.body
    if body is None:
        return [], soup

    tables = [t for t in body.find_all("table") if not t.find_parent("table")]

    docs = []
    for t in tables:
        docs.extend(table_to_docs(t, url))

    for t in tables:
        t.decompose()

    return docs, soup

In [47]:
table_docs, soup = extract_table_docs(html, "url")

table_docs

[]

In [48]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

headers_to_split_on = [("h1", "h1"), ("h2", "h2"), ("h3", "h3"), ("h4", "h4"), ("h5", "h5"), ("h6", "h6")]

html_semantic_splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    max_chunk_size=1500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", " "],
    elements_to_preserve=["table", "ul", "ol"],
    external_metadata= {"url": "the_url"}, # We add the url as we will implement RAG with Sources
    preserve_links=False,
)

def clean_chunks(semantic_chunks):
    new_chunks = []
    header_1_met = False
    share_chunk_met = False
    footer_reached = False

    for i in range(0, len(semantic_chunks)):
        chunk = semantic_chunks[i]
        share_this_page_chunk = False
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                    if not header_1_met and HEADER_KEY.match(key):
                            if key == "h1":
                                header_1_met = True
                                break

                    if not share_chunk_met and value == "Sharethis page":
                            share_chunk_met = True
                            new_chunks[-1].page_content = chunk.page_content[20:].strip()
                            share_this_page_chunk = True
                            break

                    if key == "h3" and value == "About us":
                          footer_reached = True
                          break

        # If we reached the footer, there is no need to continue
        if footer_reached:
              break

        # Skip the chunk if we have not encountered our first header yet
        if not header_1_met or share_this_page_chunk:
             continue

        # Remove meaningless encodings from the chunk page_content
        chunk.page_content.replace("\xa0", " ")

        # Add the chunk
        new_chunks.append(chunk)

    return new_chunks

def post_process_header_tags(semantic_chunks):
    # List to contain header texts of h1-h6.
    headers = []
    # Variable to hold current smallest header traversed
    curr_smallest_header_traversed = 0
    for chunk in semantic_chunks:
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                if HEADER_KEY.match(key):
                    header_index = int(key[-1])

                    # Adjust the header hierarchy for the chunk
                    if header_index == curr_smallest_header_traversed:
                        headers.pop()
                        headers.append((header_index,value))
                    elif header_index > curr_smallest_header_traversed:
                        curr_smallest_header_traversed = header_index
                        headers.append((header_index, value))
                    else:
                        curr_smallest_header_traversed = header_index
                        for i in range(1, len(headers) + 1):
                            if headers[-1][0] >= header_index:
                                headers.pop()
                            else:
                                break

                        headers.append((header_index, value))

                    chunk_header = ""

                    # Create the chunk header string
                    for header in headers:
                        chunk_header = chunk_header + header[1] + " - "

                    # Append all ancestor headers to the chunk content
                    chunk.page_content = chunk_header + chunk.page_content

def strip_links(soup):
    for a in soup.find_all("a"):
        a.unwrap()
    soup.smooth()
    return soup

In [49]:
soup = strip_links(soup)
semantic_chunks = html_semantic_splitter.split_text(str(soup))
cleaned_semantic_chunks = clean_chunks(semantic_chunks)
post_process_header_tags(cleaned_semantic_chunks)
cleaned_semantic_chunks.extend(table_docs)

cleaned_semantic_chunks

[Document(metadata={'h1': 'Residential rental agreements', 'url': 'the_url'}, page_content='Residential rental agreements - On this page: Rental agreements What should be in a rental agreement Different types of residential rental agreements Before moving in Changing agreements Conditions that cannot be in an agreement Forms you might need Other pages have information about renewing agreements, notices to vacate, and renters giving notice.'),
 Document(metadata={'h2': 'Rental agreements', 'url': 'the_url'}, page_content='Residential rental agreements - Rental agreements - A rental agreement used to be called a lease or a residential tenancy agreement. It is a contract between the renter (tenant) and rental provider (landlord). There are 3 types of rental agreements for residential properties such as houses, units and apartments. There are different agreements for rooming houses, caravan parks, and site tenants in residential parks and villages.'),
 Document(metadata={'h2': 'What should

Now our chunks are way more semantically clean and precise!